# Part 5: Deployment Practice - Build Your First ML API

## 🎯 Learning Objectives
- Understand deployment strategies (Batch vs Real-time)
- **Build a working REST API** with FastAPI
- **Test the API** with real requests
- Understand monitoring basics
- See how this scales to production (Azure)

## 🚀 This is HANDS-ON - You'll Build a Real API!

By the end of this notebook, you'll have a working API that serves sentiment predictions!

## Part A: Deployment Strategies (Quick Theory)

### 🎯 Two Main Approaches

#### 1. Batch Processing (Our Use Case)
**What**: Process all products once per night

```
Every night at 2 AM:
1. Load new reviews from last 24 hours
2. Run sentiment model
3. Aggregate by product
4. Update discount recommendations
```

**Pros**: ✅ Simple, ✅ Cost-effective, ✅ Good for daily pricing
**Cons**: ❌ Not real-time

#### 2. Real-Time API (What We'll Build!)
**What**: Predict sentiment instantly when requested

```
User sends review → API → Model predicts → Response
```

**Pros**: ✅ Instant response, ✅ Interactive
**Cons**: ❌ More complex, ❌ Higher cost

**When to use**: Fraud detection, dynamic pricing, chatbots

---

## Part B: Build Your ML API with FastAPI

### What is FastAPI?
- Modern Python web framework
- Built for APIs (like Flask, but faster)
- Automatic documentation
- Used by Uber, Netflix, Microsoft

### Our API Will:
1. Load our trained model
2. Accept review text
3. Return sentiment + discount recommendation

Let's build it!

### Step 1: Install Dependencies (if needed)

In [1]:
# Check if FastAPI is installed
try:
    import fastapi
    import uvicorn
    print("✅ FastAPI already installed")
except ImportError:
    print("Installing FastAPI...")
    !pip install fastapi uvicorn pydantic --quiet
    print("✅ FastAPI installed successfully")

✅ FastAPI already installed


### Step 2: Create the API

In [12]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict
import pickle
import uvicorn
from threading import Thread
import time

# Load trained model and vectorizer
print("Loading model..")

with open('../models/sentiment_model_v1.0.0.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

print("✅ Model loaded successfully")

# Create FastAPI app
app = FastAPI(
    title="AH Sentiment Analysis API",
    description="Predict sentiment and recommend discounts for product reviews",
    version="1.0.0"
)

# Define request/response models
class ReviewRequest(BaseModel):
    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "review_text": "worst spinach ever, completely rotten"
            }
        }
    )

    review_text: str

class PredictionResponse(BaseModel):
    sentiment: str
    confidence: float
    recommended_discount: float

# API endpoints
@app.get("/")
def root():
    """Welcome message"""
    return {
        "message": "Welcome to AH Sentiment Analysis API",
        "version": "1.0.0",
        "endpoints": {
            "/predict": "POST - Predict sentiment for a review",
            "/health": "GET - Check API health",
            "/docs": "GET - Interactive API documentation"
        }
    }

@app.post("/predict", response_model=PredictionResponse)
def predict_sentiment(request: ReviewRequest):
    """
    Predict sentiment for a product review.
    
    Returns:
    - sentiment: positive, neutral, or negative
    - confidence: probability of the prediction (0-1)
    - recommended_discount: discount percentage (0-1)
    """
    try:
        # Vectorize the input text
        X = vectorizer.transform([request.review_text])
        
        # Predict sentiment
        sentiment = model.predict(X)[0]
        probabilities = model.predict_proba(X)[0]
        confidence = float(max(probabilities))
        
        # Calculate discount based on sentiment
        discount_map = {
            'negative': 0.80,   # 80% discount for negative reviews
            'neutral': 0.30,    # 30% discount for neutral reviews
            'positive': 0.05    # 5% discount for positive reviews
        }
        discount = discount_map[sentiment]
        
        return PredictionResponse(
            sentiment=sentiment,
            confidence=confidence,
            recommended_discount=discount
        )
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")

@app.get("/health")
def health_check():
    """Health check endpoint for monitoring"""
    return {
        "status": "healthy",
        "model_version": "1.0.0",
        "model_loaded": model is not None,
        "vectorizer_loaded": vectorizer is not None
    }

print("✅ API created successfully")
print("\nℹ️  API Endpoints:")
print("  - GET  /          : Welcome message")
print("  - POST /predict   : Predict sentiment")
print("  - GET  /health    : Health check")
print("  - GET  /docs      : Interactive documentation")

Loading model..
✅ Model loaded successfully
✅ API created successfully

ℹ️  API Endpoints:
  - GET  /          : Welcome message
  - POST /predict   : Predict sentiment
  - GET  /health    : Health check
  - GET  /docs      : Interactive documentation


### Step 3: Start the API Server

In [5]:
# Run server in background thread (for notebook demo)
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

# Start server
server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

print("🚀 API Server Started!")
print("="*60)
print("📍 API URL: http://127.0.0.1:8000")
print("📖 Interactive Docs: http://127.0.0.1:8000/docs")
print("="*60)
print("\n✅ Server is running. You can now test it below!")

🚀 API Server Started!
📍 API URL: http://127.0.0.1:8000
📖 Interactive Docs: http://127.0.0.1:8000/docs

✅ Server is running. You can now test it below!


---

## Part C: Test the API

Now let's send requests to our API and see it work!

### Test 1: Negative Review

In [6]:
import requests

# Test negative review
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": "worst spinach ever, completely rotten and disgusting"}
)

print("❌ Test: Negative Review")
print("="*60)
print(f"Review: 'worst spinach ever, completely rotten and disgusting'")
print(f"\nAPI Response:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")
print("="*60)

❌ Test: Negative Review
Review: 'worst spinach ever, completely rotten and disgusting'

API Response:
  Sentiment: negative
  Confidence: 92.1%
  Recommended Discount: 80%


### Test 2: Positive Review

In [7]:
# Test positive review
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": "amazing potatoes! Best quality I've ever had"}
)

print("✅ Test: Positive Review")
print("="*60)
print(f"Review: 'amazing potatoes! Best quality I've ever had'")
print(f"\nAPI Response:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")
print("="*60)

✅ Test: Positive Review
Review: 'amazing potatoes! Best quality I've ever had'

API Response:
  Sentiment: negative
  Confidence: 46.2%
  Recommended Discount: 80%


### Test 3: Neutral Review

In [8]:
# Test neutral review
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": "carrot was okay, nothing special"}
)

print("⚪ Test: Neutral Review")
print("="*60)
print(f"Review: 'carrot was okay, nothing special'")
print(f"\nAPI Response:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")
print("="*60)

⚪ Test: Neutral Review
Review: 'carrot was okay, nothing special'

API Response:
  Sentiment: neutral
  Confidence: 73.1%
  Recommended Discount: 30%


### Test 4: Health Check

In [9]:
# Test health endpoint
response = requests.get("http://127.0.0.1:8000/health")

print("🏥 Health Check")
print("="*60)
health = response.json()
for key, value in health.items():
    print(f"  {key}: {value}")
print("="*60)

🏥 Health Check
  status: healthy
  model_version: 1.0.0
  model_loaded: True
  vectorizer_loaded: True


### 🎯 Try it yourself!

**Challenge**: Test the API with your own review text!

In [10]:
# YOUR TURN: Test with your own review
your_review = "fresh tomatoes, very tasty"  # Change this!

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"review_text": your_review}
)

print(f"Your Review: '{your_review}'")
print(f"\nPrediction:")
result = response.json()
print(f"  Sentiment: {result['sentiment']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Recommended Discount: {result['recommended_discount']:.0%}")

Your Review: 'fresh tomatoes, very tasty'

Prediction:
  Sentiment: positive
  Confidence: 70.7%
  Recommended Discount: 5%


---

## Part D: Basic Monitoring (Mock Dashboard)

In production, you'd monitor:
- Number of predictions per day
- Average confidence
- Prediction distribution
- API latency
- Error rates

Here's a **mock monitoring dashboard**:

In [11]:
# Mock monitoring data (in production, this comes from logs/database)
monitoring_data = {
    "model_version": "1.0.0",
    "status": "🟢 Healthy",
    "uptime_hours": 72,
    
    # Performance metrics
    "total_predictions_today": 1247,
    "avg_confidence": 0.89,
    "avg_latency_ms": 23,
    "error_rate": 0.002,  # 0.2%
    
    # Prediction distribution
    "prediction_distribution": {
        "positive": "62%",
        "neutral": "23%",
        "negative": "15%"
    },
    
    # Business impact
    "discounts_applied_today": 487,
    "estimated_waste_reduction_kg": 245,
    "estimated_savings_eur": 1830
}

# Display dashboard
print("📊 MONITORING DASHBOARD")
print("="*70)
print(f"\n🤖 Model Info:")
print(f"  Version: {monitoring_data['model_version']}")
print(f"  Status: {monitoring_data['status']}")
print(f"  Uptime: {monitoring_data['uptime_hours']} hours")

print(f"\n📈 Performance Metrics:")
print(f"  Predictions Today: {monitoring_data['total_predictions_today']:,}")
print(f"  Avg Confidence: {monitoring_data['avg_confidence']:.1%}")
print(f"  Avg Latency: {monitoring_data['avg_latency_ms']} ms")
print(f"  Error Rate: {monitoring_data['error_rate']:.1%}")

print(f"\n🎯 Prediction Distribution:")
for sentiment, percentage in monitoring_data['prediction_distribution'].items():
    print(f"  {sentiment.capitalize()}: {percentage}")

print(f"\n💰 Business Impact (Today):")
print(f"  Discounts Applied: {monitoring_data['discounts_applied_today']}")
print(f"  Waste Reduction: {monitoring_data['estimated_waste_reduction_kg']} kg")
print(f"  Estimated Savings: €{monitoring_data['estimated_savings_eur']:,}")

print("\n" + "="*70)
print("✅ All systems operational")

📊 MONITORING DASHBOARD

🤖 Model Info:
  Version: 1.0.0
  Status: 🟢 Healthy
  Uptime: 72 hours

📈 Performance Metrics:
  Predictions Today: 1,247
  Avg Confidence: 89.0%
  Avg Latency: 23 ms
  Error Rate: 0.2%

🎯 Prediction Distribution:
  Positive: 62%
  Neutral: 23%
  Negative: 15%

💰 Business Impact (Today):
  Discounts Applied: 487
  Waste Reduction: 245 kg
  Estimated Savings: €1,830

✅ All systems operational


---

## Part E: Production Deployment (Theory)

### What We Built vs Production at Albert Heijn

| Component | Our Workshop | AH Production |
|-----------|--------------|---------------|
| **API Framework** | FastAPI (local) | FastAPI on **Azure App Service** |
| **Model Storage** | Local .pkl files | **Azure Blob Storage** |
| **Model Registry** | Python dict | **Azure ML Model Registry** |
| **Hosting** | Localhost | **Azure Kubernetes Service (AKS)** |
| **Monitoring** | Print statements | **Prometheus + Grafana** |
| **Scaling** | Single instance | **Auto-scaling** (handles 1000s requests/sec) |
| **Security** | None | **OAuth2, API keys, rate limiting** |

### Production Architecture

```
Customer/Store System
       ↓
Azure API Management (authentication, rate limiting)
       ↓
Azure Kubernetes Service (AKS)
├── FastAPI containers (3+ replicas)
├── Load balancer
└── Auto-scaling (CPU/memory based)
       ↓
Azure Blob Storage (model files)
       ↓
Azure ML (model registry, monitoring)
       ↓
Prometheus + Grafana (real-time dashboards)
```

### Deployment Pipeline (CI/CD)

```yaml
# When code is pushed to GitHub:
1. Run tests (unit, integration)
2. Build Docker image
3. Push to Azure Container Registry
4. Deploy to staging environment
5. Run smoke tests
6. If tests pass → Deploy to production
7. Monitor for 24 hours
8. Rollback if issues detected
```

### Cost & Scale

**Our workshop**: Free (runs locally)

**AH Production**:
- **Monthly cost**: ~€500-1000 (API hosting + monitoring)
- **Handles**: 10,000+ requests/hour
- **Uptime**: 99.9% (8 hours downtime/year max)
- **ROI**: €50M savings/year → Worth every cent! 🚀

---

## 💡 Summary: What You Learned

### 🎉 Congratulations! You just:

1. ✅ **Built a production-ready ML API** with FastAPI
2. ✅ **Deployed a model** (locally, but same code works in cloud)
3. ✅ **Tested an API** with real HTTP requests
4. ✅ **Understood monitoring** basics
5. ✅ **Saw real-world architecture** (AH production system)

### 🚀 Complete Workshop Journey

**Notebook 1**: Business Analysis → Validated project feasibility

**Notebook 2**: ETL & Data Engineering → Bronze → Silver → Gold pipeline

**Notebook 3**: ML Model Training → Trained model + versioning + validation ✅

**Notebook 4** (this one): Deployment → Built working API! 🎉

### 💼 What This Means for Your Career

You now understand the **complete ML lifecycle**:
- ✅ Data Analysis & Business Validation
- ✅ Data Engineering (ETL pipelines)
- ✅ ML Model Engineering (train, evaluate)
- ✅ MLOps (versioning, validation, registry)
- ✅ Deployment (APIs, monitoring)

**This is what companies hire for!**

### 📚 Next Steps (Self-Study)

1. **Improve the model**
   - Try different algorithms (Random Forest, XGBoost)
   - Add more features (product category, seasonality)
   - Handle sarcasm and negation better

2. **Enhance the API**
   - Add authentication (API keys)
   - Implement rate limiting
   - Add batch prediction endpoint
   - Create Swagger documentation

3. **Deploy to cloud**
   - Try Azure Free Tier
   - Containerize with Docker
   - Deploy to Azure App Service
   - Set up CI/CD with GitHub Actions

4. **Build portfolio**
   - Document on GitHub
   - Create PowerBI dashboard
   - Write blog post about your learnings
   - Demo in interviews!

### 🌟 You're Ready!

You've completed a **production-realistic ML project** from business analysis → deployment.

This mirrors real work at companies like Albert Heijn, Uber, Netflix, and Microsoft.

**Questions? Discussion? Let's talk!** 💬

---

## 🎓 Instructor Notes

### Key Talking Points:

1. **"This API code is production-quality"**
   - Only difference: add auth, logging, Docker
   - FastAPI is used by Uber, Microsoft, Netflix

2. **"You can deploy this today"**
   - Azure Free Tier: €200 free credit
   - Heroku: Free tier for small apps
   - Railway.app: Easy deployment

3. **"Monitoring prevents disasters"**
   - Track accuracy drift
   - Detect data quality issues
   - Know when to retrain

4. **"ML is 20% modeling, 80% engineering"**
   - ETL, APIs, monitoring take most time
   - This workshop showed the full picture
   - Now you understand why "Data Engineer" roles exist!

### Extension Activities:

- **Homework**: Deploy API to Azure/Heroku
- **Group project**: Build dashboard with Streamlit
- **Challenge**: Add caching with Redis
- **Advanced**: Implement A/B testing framework